# Intelligent Form Agent -- Demonstration Runs

Three required example runs for demonstration purposes:

1. Answering a question from a single form
2. Generating a summary of one form
3. Providing a holistic answer across multiple forms

Each cell below is one real conversational turn, and all cells share one `thread_id`, so this is a genuine multi-turn conversation, not isolated snippets.

### Why a notebook instead of the CLI?
This notebook calls the exact same underlying functions (`build_turn_update()` + `graph.invoke()`) that `main.py` uses when you run it as `python main.py data/form1.png data/form2.png` and then type questions at the `You:` prompt. Functionally this is the identical agent, behaving the same way. The notebook format is used here specifically because it's a cleaner way to show narration, code, and real output together in one reviewable document, rather than a raw terminal transcript.

See `docs/architecture_walkthrough.md` for how a turn flows through the system, and `technical_considerations.md` for the design decisions behind it.

In [6]:
import sys
import textwrap
import uuid
from pathlib import Path

from dotenv import load_dotenv

# Project root on sys.path regardless of Jupyter's working directory
# -- same fix as scripts/try_extraction.py and main.py.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
print(f"Project root: {PROJECT_ROOT}")

load_dotenv(PROJECT_ROOT / ".env")

from src.graph import build_graph
from src.turn_input import build_turn_update


Project root: /Users/eduardo/projects/intelligent-form-agent


## Configuration


In [7]:
DANIEL_PATH = str(PROJECT_ROOT / "data" / "sample_daniel.png")
DANIELLE_PATH = str(PROJECT_ROOT / "data" / "sample_danielle.png")
JAMES_PATH = str(PROJECT_ROOT / "data" / "sample_pdf_james.pdf")


In [8]:
def print_wrapped(label, text, width=90):
    """Prints a labeled line, wrapped to `width` chars.

    Plain print() of a long LLM response renders as one very long
    line in a notebook cell -- readable only by scrolling
    sideways. This wraps it the way a terminal naturally would.
    """
    text = text or "(no response)"
    wrapped = textwrap.fill(
        text,
        width=width,
        initial_indent=f"{label}: ",
        subsequent_indent=" " * (len(label) + 2),
    )
    print(wrapped)


graph = build_graph()
config = {"configurable": {"thread_id": f"demo-{uuid.uuid4()}"}}


def run_turn(**kwargs):
    """Builds one turn's update, invokes the graph, prints the result.

    Same two-line pattern main.py's _run_turn() uses -- one cell is
    one conversational turn, all sharing the same thread_id above.
"""
    update = build_turn_update(**kwargs)
    result = graph.invoke(update, config=config)
    print_wrapped("Assistant", result.get("response"))
    if result.get("needs_escalation"):
        print_wrapped(
            "Flagged", result.get("escalation_reason")
        )
    print()
    return result


## Ingest the sample forms

All three demonstrations below use these same three forms, ingested once.

In [9]:
run_turn(
    uploaded_files=[DANIEL_PATH, DANIELLE_PATH, JAMES_PATH]
);


Assistant: Ingested 3 form(s): sample_daniel.png, sample_danielle.png,
           sample_pdf_james.pdf#page1.



## Demonstration 1: Answering a question from a single form

In [10]:
result_1 = run_turn(
    user_text=(
        "What is Daniel's date of birth, and what services "
        "are being requested for him?"
    )
)


Assistant: Daniel's date of birth is March 20, 1992. The services being requested for him
           are open placement (44300) for aftercare following joint replacement surgery
           from November 20, 2022, to November 26, 2022, and needle catheter (44015) for
           Type 1 diabetes mellitus with other specified complication on November 29,
           2023.



## Demonstration 2: Generating a summary of one form

In [ ]:
result_2 = run_turn(
    user_text=("Now please summarize Danielle's form. Hide her last name from the summary. "
               " Do the same (hide last names) for any other people referenced in the report"
    )
)


Assistant: Danielle, born on October 3, 1984, has a non-urgent request for two services.
           The first is a suture procedure for an unspecified fracture of the right tibia
           with delayed healing, scheduled from June 2 to June 8, 2023. The second is
           athletic training for arthropathies in the ankle and foot, scheduled from
           October 22 to October 26, 2023. The requesting provider is Jeremy, and the
           service provider is Sarah.



## Demonstration 3: Holistic answer across multiple forms

In [15]:
result_3 = run_turn(
    user_text=(
        "Compare all three forms -- Daniel, Danielle, and James. "
        "What do they have in common, and how do they differ, "
        "particularly around urgency and diagnosis? "
        "Remember not to display their last names."
    )
)


Assistant: **Commonalities:** - All three forms involve requests for medical services with
           specific diagnosis codes. - Each form specifies a requesting provider and a
           service provider.  **Differences:** - **Urgency:**   - Daniel and Danielle's
           requests are classified as non-urgent.   - James's request is classified as
           urgent.  - **Diagnosis and Services:**   - **Daniel:** Services include open
           placement for aftercare following joint replacement surgery and a needle
           catheter for Type 1 diabetes mellitus with complications.   - **Danielle:**
           Services include a suture procedure for a delayed healing fracture of the right
           tibia and athletic training for arthropathies in the ankle and foot.   -
           **James:** Services include surgical laparoscopy for subluxation of the
           acromioclavicular joint and an X-ray for atheroembolism of the right upper
           extremity.  These differences hig

## Wrap-up

All three demonstrations ran as turns in one continuous conversation (shared `thread_id`), the same mechanism the CLI uses for multi-turn memory.

For adversarial testing of the escalation policy against deliberately broken forms, see `adversarial_escalation_tests.ipynb`.